<a href="https://colab.research.google.com/github/AmishaOnMain/Personal_Finance_Advisor/blob/main/Personal_Finance_Advisor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y google-adk

!pip install -qU langchain langchain-community pypdf

!pip install -qU \
langchain \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
langchain-chroma \
sentence-transformers \
chromadb


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

!wget -O sebi.pdf "https://investor.sebi.gov.in/pdf/downloadable-documents/Financial%20Education%20Booklet%20-%20English.pdf"
!wget -O gst.pdf "https://a2ztaxcorp.net/wp-content/uploads/2025/09/CBIC-GST-Ready-Reckoner-indicating-updated-Central-Goods-and-Services-Tax-CGST-rates-on-goods.pdf"

from langchain_community.document_loaders import PyPDFLoader

all_documents = []

for file in ["sebi.pdf", "gst.pdf"]:
    loader = PyPDFLoader(file)
    docs = loader.load()
    all_documents.extend(docs)
    print(f"Loaded {len(docs)} pages from {file}")




In [ ]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(all_documents)
print(f"Total chunks: {len(chunks)}")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

vector_store = Chroma(
    collection_name="finance_docs",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db"
)
vector_store.add_documents(documents=chunks)
print("Knowledge base ready!")

In [ ]:
results = vector_store.similarity_search("What is the GST rate on laptops?", k=3)
for doc in results:
    source = doc.metadata.get('source', 'Unknown')
    page = doc.metadata.get('page', 'N/A')
    print(f"Source: {source} (Page {page + 1})")
    print(f"Content: {doc.page_content[:200]}...")
    print()

In [ ]:
from langchain.tools import tool

@tool
def search_finance_docs(question: str) -> str:
    """Search official Indian government financial documents for rules, regulations, tax rates, investment guidance, and financial education content."""
    results = vector_store.similarity_search(question, k=3)
    if not results:
        return "No relevant information found in the knowledge base."
    context = ""
    for doc in results:
        source = doc.metadata.get('source', 'Unknown')
        page = doc.metadata.get('page', 'N/A')
        context += f"Source: {source} (Page {page + 1})\n"
        context += f"Content: {doc.page_content}\n\n"
    return context

In [ ]:
import requests

url = "https://api.gold-api.com/price/XAU/INR"
response = requests.get(url)
data = response.json()

price_per_gram = data["price"] / 31.1035
print(f"Gold: ₹{price_per_gram:.2f} per gram")

In [ ]:
@tool
def get_market_price(asset: str) -> str:
    """Get current market price for a financial asset in Indian Rupees. Pass one of: gold, silver."""
    symbols = {
        "gold": "XAU",
        "silver": "XAG",
    }
    symbol = symbols.get(asset.lower())
    if not symbol:
        return "Supported assets: gold, silver"

    url = f"https://api.gold-api.com/price/{symbol}/INR"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        price = data.get("price")
        if price:
            price_per_gram = price / 31.1035
            return f"{asset.title()}: ₹{price_per_gram:.2f} per gram (₹{price:.2f} per troy ounce). Source: Gold-API"
    return f"Could not fetch price for {asset}"

In [ ]:
!pip install -qU langchain-google-genai

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')

model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key=api_key,
)

In [ ]:
system_prompt = """You are a Personal Finance AI Advisor for Indian citizens.

You have access to these tools:
- search_finance_docs: Search official Indian government documents for GST rates,
  investment guidance, tax saving options, and financial education
- get_market_price: Get live market prices for gold and silver in Indian Rupees

Help the user by looking up the relevant data using your tools and giving clear,
specific answers with actual numbers and rates. Always mention the source of
your information. All monetary values should be in Indian Rupees (₹) unless
specified otherwise.
"""

agent = create_agent(
    model=model,
    tools=[search_finance_docs, get_market_price],
    system_prompt=system_prompt,
)

In [ ]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "What is the current gold price?"}]
})
print(response["messages"][-1].content)

response = agent.invoke({
    "messages": [{"role": "user", "content": "I want to buy a gold chain. What's the current gold price and how much GST will I pay?"}]
})
print(response["messages"][-1].content)

In [ ]:
!pip install -qU gradio

In [ ]:
import gradio as gr

def finance_advisor(question):
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return response["messages"][-1].content

demo = gr.Interface(
    fn=finance_advisor,
    inputs=gr.Textbox(lines=2, placeholder="Ask a finance question...", label="Question"),
    outputs=gr.Textbox(lines=10, label="Answer"),
    title="Personal Finance AI Advisor",
    description="Ask about gold prices, silver prices, GST rates, tax saving options, and more.",
)
demo.launch(debug=True)

In [ ]:
!pip install -qU assemblyai

In [ ]:
from google.colab import userdata
import assemblyai as aai
import requests
import json
import tempfile

ASSEMBLYAI_API_KEY = userdata.get('ASSEMBLYAI_API_KEY')
MURF_API_KEY = userdata.get('MURF_API_KEY')

aai.settings.api_key = ASSEMBLYAI_API_KEY

def speech_to_text(audio_path):
    """Converts an audio file to text using AssemblyAI."""
    transcriber = aai.Transcriber()
    config = aai.TranscriptionConfig(
        speech_models=["universal-3-pro", "universal-2"],
        language_detection=True,
        speaker_labels=True,
    )
    transcript = transcriber.transcribe(audio_path, config=config)
    return transcript.text if transcript.text else ""

def text_to_speech(text):
    """Converts text to an MP3 audio file using Murf.AI Falcon."""
    url = "https://global.api.murf.ai/v1/speech/stream"
    payload = {
        "text": text,
        "voiceId": "en-US-natalie",
        "model": "FALCON",
        "multiNativeLocale": "en-US",
        "sampleRate": 24000,
        "format": "MP3",
    }
    headers = {
        "Content-Type": "application/json",
        "api-key": MURF_API_KEY,
    }
    response = requests.post(url, headers=headers, data=json.dumps(payload))
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    temp_file.write(response.content)
    temp_file.close()
    return temp_file.name

In [ ]:
def finance_advisor_voice(audio):
    if audio is None:
        return "No audio recorded.", "Please record your question first.", None
    question = speech_to_text(audio)
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content
    audio_path = text_to_speech(answer)
    return question, answer, audio_path

In [ ]:
voice_demo = gr.Interface(
    fn=finance_advisor_voice,
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath", label="Ask your question"),
    outputs=[
        gr.Textbox(label="Your Question (transcribed)"),
        gr.Textbox(lines=10, label="Answer"),
        gr.Audio(label="Answer (audio)"),
    ],
    title="Personal Finance AI Advisor (Voice)",
    description="Speak your finance question and get both a text and audio response.",
)

voice_demo.launch(debug=True, share=True)